# QWEN INFERENCE

In [ ]:
import torch
import transformers
from datasets import load_dataset
import os
import random
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import librosa
import numpy as np
import json
from google.colab import drive
drive.mount('/content/drive')

random.seed(42)

DRIVE_DIR = "/content/drive/MyDrive/TextMiningProject/"
DATASET = os.path.join(DRIVE_DIR, "data")
os.makedirs(DATASET, exist_ok=True)

Mounted at /content/drive


In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    token = userdata.get('HF_TOKEN')
    login(token)
    print("Logged in via Colab Secret")
except:
    login()

Logged in via Colab Secret


In [ ]:
ALL_DATA_JSON = os.path.join(DATASET, "all_data_processed.json")

with open(ALL_DATA_JSON, "r", encoding="utf-8") as f:
    all_data = json.load(f)
    samsum_data = [item for item in all_data if item["dataset"] == "samsum"]
    xsum_data   = [item for item in all_data if item["dataset"] == "xsum"]
    print(f"Loaded all_data from {ALL_DATA_JSON}")
    print(f"Total samples: {len(all_data)}")

Loaded all_data from /content/drive/MyDrive/TextMiningProject/data/all_data_processed.json
Total samples: 600


## Model: Qwen2.5-Omni-3B

In [ ]:
!pip install -q qwen-omni-utils[decord] -U
!pip install -U bitsandbytes>=0.46.1 accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 115.2 MB/s eta 0:00:00


In [ ]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [ ]:
from qwen_omni_utils import process_mm_info
from transformers import (
    Qwen2_5OmniThinkerForConditionalGeneration,
    Qwen2_5OmniProcessor,
)

model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-Omni-3B",
    device_map="auto",
    trust_remote_code=True,
    quantization_config=bnb_config,
    attn_implementation="sdpa"
)

processor = Qwen2_5OmniProcessor.from_pretrained(
    "Qwen/Qwen2.5-Omni-3B",
    trust_remote_code=True,
)


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1442 [00:00<?, ?it/s]

Qwen2_5OmniThinkerForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-3B
Key                                                                                                      | Status     |  | 
---------------------------------------------------------------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.transformer_blocks.{0...21}.ff.ff.{0, 3}.weight                             | UNEXPECTED |  | 
token2wav.code2wav_bigvgan_model.resblocks.{0...17}.convs2.{0, 1, 2}.weight                              | UNEXPECTED |  | 
talker.model.layers.{0...23}.self_attn.k_proj.weight                                                     | UNEXPECTED |  | 
token2wav.code2wav_bigvgan_model.resblocks.{0...17}.activations.{0, 1, 2, 3, 4, 5}.act.alpha             | UNEXPECTED |  | 
token2wav.code2wav_bigvgan_model.resblocks.{0...17}.convs1.{0, 1, 2}.weight                              | UNEXPECTED |  | 
token2wav.code2wav_dit_model.proj_out.bias        

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

In [ ]:
SYS_PROMPT = "You are a helpful summarization assistant."
PROMPT_AUDIO = "Summarize this audio in one sentence."
PROMPT_TEXT = "Summarize this text in one sentence."

In [ ]:
import gc
import torch

def do_qwen_inference(
    model,
    processor,
    batch_data,
    sys_prompt="",
    prompt="",
    max_new_tokens=64,
    decoding="temperature",
    use_audio_in_video=True
):
    messages_batch = []
    if prompt == PROMPT_AUDIO:
        for item in batch_data:
            message = [
                {
                    "role": "system",
                    "content": [{"type": "text", "text": sys_prompt}],
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "audio", "audio": item["audio_path"]},
                        {"type": "text", "text": prompt},
                    ],
                },
            ]
            messages_batch.append(message)
    else:
        for item in batch_data:
            message = [
                {
                    "role": "system",
                    "content": [{"type": "text", "text": sys_prompt}],
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": item["text"]},
                        {"type": "text", "text": prompt},
                    ],
                },
            ]
            messages_batch.append(message)

    text = processor.apply_chat_template(
        messages_batch,
        add_generation_prompt=True,
        tokenize=False
    )

    if prompt == PROMPT_AUDIO:
        audios, images, videos = process_mm_info(
            messages_batch,
            use_audio_in_video=use_audio_in_video
        )

        inputs = processor(
            text=text,
            audio=audios,
            images=images,
            videos=videos,
            return_tensors="pt",
            padding=True,
            use_audio_in_video=use_audio_in_video
        ).to(model.device).to(model.dtype)

    elif prompt == PROMPT_TEXT:
        inputs = processor(text=text, return_tensors="pt", padding=True)
        inputs = inputs.to(model.device).to(model.dtype)


    # Decoding strategies
    if decoding == "temperature":
        text_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=1.0,
            top_p=0.95,
            top_k=64,
            use_audio_in_video=use_audio_in_video
        )
    elif decoding == "greedy":
        text_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_audio_in_video=use_audio_in_video
        )
    elif decoding == "beam":
        text_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=2,
            do_sample=False,
            early_stopping=True,
            use_audio_in_video=use_audio_in_video
        )
    else:
        raise ValueError("Invalid decoding mode. Use: 'temperature', 'greedy', or 'beam'.")

    summaries = processor.batch_decode(
        text_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    if prompt == PROMPT_AUDIO:
        del inputs, text_ids, audios, images, videos
    else:
        del inputs, text_ids
    torch.cuda.empty_cache()
    gc.collect()

    return summaries


#### qwen audio summarization

- temperature

In [ ]:
batch_size = 2

for i in tqdm(range(0, len(all_data), batch_size), desc="Generating Audio Summary (Qwen)"):
    batch = all_data[i:i+batch_size]

    results = do_qwen_inference(
        model=model,
        processor=processor,
        batch_data=batch,
        sys_prompt=SYS_PROMPT,
        prompt=PROMPT_AUDIO,
        decoding="temperature"
    )

    for item, summary in zip(batch, results):
        item["summary_qwen_audio"] = summary

output_dir = "/content/drive/MyDrive/TextMiningProject/qwen"
output_json = "/content/drive/MyDrive/TextMiningProject/qwen/audio_summaries_temperature.json"

os.makedirs(output_dir, exist_ok=True)

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

print(f"\nSaved results to {output_json}")

Generating Audio Summary (Qwen):  15%|█▍        | 44/300 [13:44<1:17:21, 18.13s/it]WARNING:root:System prompt modified, audio output may not work as expected. Audio output mode only works when using default system prompt 'You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving auditory and visual inputs, as well as generating text and speech.'


- greedy

In [ ]:
batch_size = 1

for i in tqdm(range(0, len(all_data), batch_size), desc="Generating Audio Summary (Qwen)"):
    batch = all_data[i:i+batch_size]

    results = do_qwen_inference(
        model=model,
        processor=processor,
        batch_data=batch,
        sys_prompt=SYS_PROMPT,
        prompt=PROMPT_AUDIO,
        decoding="greedy"
    )

    for item, summary in zip(batch, results):
        item["summary_qwen_audio"] = summary

output_dir = "/content/drive/MyDrive/TextMiningProject/qwen"
output_json = "/content/drive/MyDrive/TextMiningProject/qwen/audio_summaries_greedy.json"

os.makedirs(output_dir, exist_ok=True)

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

print(f"\nSaved results to {output_json}")

Generating Audio Summary (Qwen): 100%|██████████| 600/600 [1:35:54<00:00,  9.59s/it]


Saved results to /content/drive/MyDrive/TextMiningProject/qwen/audio_summaries_greedy.json


- beam

In [ ]:
batch_size = 1

for i in tqdm(range(0, len(all_data), batch_size), desc="Generating Audio Summary (Qwen)"):
    batch = all_data[i:i+batch_size]

    results = do_qwen_inference(
        model=model,
        processor=processor,
        batch_data=batch,
        sys_prompt=SYS_PROMPT,
        prompt=PROMPT_AUDIO,
        decoding="beam"
    )

    for item, summary in zip(batch, results):
        item["summary_qwen_audio"] = summary

output_dir = "/content/drive/MyDrive/TextMiningProject/qwen"
output_json = "/content/drive/MyDrive/TextMiningProject/qwen/audio_summaries_beam.json"

os.makedirs(output_dir, exist_ok=True)

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

print(f"\nSaved results to {output_json}")

#### qwen text summarization

- temperature

In [ ]:
batch_size = 40

for i in tqdm(range(0, len(all_data), batch_size), desc="Generating Text Summary (Qwen)"):
    batch = all_data[i:i+batch_size]

    results = do_qwen_inference(
        model=model,
        processor=processor,
        batch_data=batch,
        sys_prompt=SYS_PROMPT,
        prompt=PROMPT_TEXT,
        decoding="temperature"
    )

    for item, summary in zip(batch, results):
        item["summary_qwen_text"] = summary

output_dir = "/content/drive/MyDrive/TextMiningProject/qwen"
output_json = "/content/drive/MyDrive/TextMiningProject/qwen/text_summaries_temperature.json"

os.makedirs(output_dir, exist_ok=True)

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

print(f"\nSaved results to {output_json}")

Generating Text Summary (Qwen): 100%|██████████| 15/15 [05:39<00:00, 22.62s/it]


Saved results to /content/drive/MyDrive/TextMiningProject/qwen/text_summaries_temperature.json


- greedy

In [ ]:
batch_size = 40

for i in tqdm(range(0, len(all_data), batch_size), desc="Generating Text Summary (Qwen)"):
    batch = all_data[i:i+batch_size]

    results = do_qwen_inference(
        model=model,
        processor=processor,
        batch_data=batch,
        sys_prompt=SYS_PROMPT,
        prompt=PROMPT_TEXT,
        decoding="greedy"
    )

    for item, summary in zip(batch, results):
        item["summary_qwen_text"] = summary

output_dir = "/content/drive/MyDrive/TextMiningProject/qwen"
output_json = "/content/drive/MyDrive/TextMiningProject/qwen/text_summaries_greedy.json"

os.makedirs(output_dir, exist_ok=True)

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

print(f"\nSaved results to {output_json}")

Generating Text Summary (Qwen): 100%|██████████| 15/15 [05:30<00:00, 22.04s/it]


Saved results to /content/drive/MyDrive/TextMiningProject/qwen/text_summaries_greedy.json


- beam

In [ ]:
batch_size = 30

for i in tqdm(range(0, len(all_data), batch_size), desc="Generating Text Summary (Qwen)"):
    batch = all_data[i:i+batch_size]

    results = do_qwen_inference(
        model=model,
        processor=processor,
        batch_data=batch,
        sys_prompt=SYS_PROMPT,
        prompt=PROMPT_TEXT,
        decoding="beam"
    )

    for item, summary in zip(batch, results):
        item["summary_qwen_text"] = summary

output_dir = "/content/drive/MyDrive/TextMiningProject/qwen"
output_json = "/content/drive/MyDrive/TextMiningProject/qwen/text_summaries_beam.json"

os.makedirs(output_dir, exist_ok=True)

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

print(f"\nSaved results to {output_json}")

Generating Text Summary (Qwen): 100%|██████████| 15/15 [08:47<00:00, 35.19s/it]


Saved results to /content/drive/MyDrive/TextMiningProject/qwen/text_summaries_beam.json
